# Experiment 1 · Full-supervised foundation models (LoRA, f = 100%)

This experiment fills the **foundation, full-supervised** block of Table 1: the four promptable foundation models (SAM2, MedSAM, MedSAM-2, SAM3) are **LoRA-fine-tuned on 100% of the training split** (`fraction = 1.0`) and evaluated **box-prompted** with the tight oracle GT box. It is the full-supervised *ceiling* for the promptable family, and it is directly comparable to the traditional full-supervised block (Exp 1) — a like-for-like full-vs-full comparison.

Every knob matches the few-shot LoRA protocol (Exp 3) exactly — LoRA `r=16, alpha=32`, `dice_bce_50_50` loss, Adam `lr=1e-4, wd=1e-4`, cosine schedule, 100 epochs, patience 15, batch size 4, bfloat16 autocast, **seed 42** — so the `f=100%` point is a valid extension of the data-efficiency curve. The only difference from the few-shot runner is the training-set size (100% of patients vs a sampled fraction). SAM (original ViT-H) is excluded from LoRA adaptation, matching Exp 3.

In [ ]:
# Move to the repository root (the directory that holds pyproject.toml) so that
# `thyroidbench` is importable and the default relative paths resolve.
import os
from pathlib import Path

here = Path.cwd()
while not (here / "pyproject.toml").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("Repo root:", Path.cwd())

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Prerequisites

- The `thyroidbench` package must be importable (install the repo, e.g. `pip install -e .`).
- The pretrained foundation-model weights must be present under `pretrained_models/` (SAM2 `sam2.1_hiera_large.pt`, MedSAM `medsam_vit_b.pth`, MedSAM-2 `MedSAM2_latest.pt`, SAM3). LoRA adapters are trained on top of these frozen backbones.
- The processed datasets and patient-level splits must be under `data/` (`data/processed`, `data/splits`).
- Training writes to Weights & Biases (project `thyroidbench`); run `wandb login` or set `WANDB_MODE=offline` first.

## The adaptation recipe

One recipe is used for every foundation model, every dataset and every fraction in
this benchmark; the *only* thing that changes across Experiments 1, 3 and 4 is the
sampled training fraction. Keeping it fixed is what makes the numbers comparable
across cells, and it is the piece most worth reusing elsewhere:

| Knob | Value |
|---|---|
| LoRA rank / alpha / dropout | `r=16`, `alpha=32`, `dropout=0.1` |
| LoRA target modules | `attn.qkv`, `attn.proj` (image encoder / ViT trunk only) |
| Frozen | everything else, including the prompt encoder and mask decoder |
| Loss | Dice + BCE, 50/50 |
| Optimiser | Adam, `lr=1e-4`, `weight_decay=1e-4` |
| Schedule | cosine annealing over `max_epochs` |
| Epochs / early stop | 100, patience 15 on val DSC (`min_delta=1e-3`) |
| Batch size | 4 |
| Precision | bfloat16 autocast |
| Prompt | tight oracle box from the ground-truth mask |
| Seed | 42 (splits, sampling, init, shuffling) |

No per-model or per-dataset hyperparameter search is run for the foundation models,
so no cell of the reported table gets a tuning advantage over another.
The config itself lives in `thyroidbench/lora.py` (`_LORA_CONFIG`).


## Run

16 runs = 4 models × 4 datasets, each LoRA-fine-tuned at `fraction = 1.0` (100% of the training split) and evaluated on the held-out test split with the tight oracle box. Each run writes `per_image_results.csv` and `summary_metrics.csv` into its own `results/fullsup_<model>_<dataset>/` folder.

These are full training runs (100 epochs, early-stop patience 15) — budget GPU time accordingly. `fraction = 1.0` and `seed = 42` are fixed inside `run.py`; there is no `--fraction` flag to pass.

In [ ]:
for model in ['sam2', 'medsam', 'medsam2', 'sam3']:
    for dataset in ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']:
        !python experiments/exp1_fullsupervised/foundation_lora/run.py --model {model} --dataset {dataset}

## Results

The aggregation step writes a single aggregate table,
`results/exp1_foundation_fullsup_summary.csv` — one row per (model, dataset)
with the oracle-box test metrics (mean DSC, IoU, precision, recall, HD95) for
the Table 1 foundation full-supervised block. It is rebuilt from the per-run
`fullsup_<model>_<dataset>/summary_metrics.csv` folders by `aggregate_fullsup.py`.

In [ ]:
import pandas as pd

summary = pd.read_csv(
    'experiments/exp1_fullsupervised/foundation_lora/results/exp1_foundation_fullsup_summary.csv')
cols = ['model', 'dataset', 'fraction', 'n_images',
        'dsc', 'iou', 'precision', 'recall', 'hd95']
summary[cols].round(4)